# Assignment 2: Sentiment Analysis using NLP & ML Models

## Source: Kaggle (IMDb Dataset)

## Objective:
To build an end-to-end Sentiment Analysis system using NLP preprocessing, feature engineering, and multiple ML models. Compare their performance using evaluation metrics.

In [3]:
# Import Libraries
import pandas as pd
import numpy as np
import re
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### 1. Data Understanding


###Load dataset

In [4]:
train = pd.read_csv("/content/Train.csv")
test = pd.read_csv("/content/Test.csv")
valid = pd.read_csv("/content/Valid.csv")

df = pd.concat([train, test, valid], ignore_index=True)

df.head()

,text,label
0,I grew up (b. 1965) watching and loving the Th...,0
1,"When I put this movie in my DVD player, and sa...",0
2,Why do people who do not know what a particula...,0
3,Even though I have great interest in Biblical ...,0
4,Im a die hard Dads Army fan and nothing will e...,1


###  Data Understanding

In [5]:
print("Shape:", df.shape)
df.info()

Shape: (50000, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    50000 non-null  object
 1   label   50000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 781.4+ KB


In [6]:
df = df.rename(columns={
    'text': 'review',
    'label': 'sentiment'
})
print("\nSentiment Distribution:\n", df['sentiment'].value_counts())


Sentiment Distribution:
 sentiment
0    25000
1    25000
Name: count, dtype: int64


In [7]:
print("\nMissing Values:\n", df.isnull().sum())



Missing Values:
 review       0
sentiment    0
dtype: int64


### Observation:
- Dataset contains text and sentiment labels.
- Some missing values may be present in text column.
- Data needs cleaning before applying NLP.

###  Data Cleaning

In [8]:
# Drop missing values
df = df.dropna()

In [9]:
df = df.rename(columns={
    'text': 'review',
    'label': 'sentiment'
})

In [10]:
df = df[['review', 'sentiment']]

In [11]:
df.head()

,review,sentiment
0,I grew up (b. 1965) watching and loving the Th...,0
1,"When I put this movie in my DVD player, and sa...",0
2,Why do people who do not know what a particula...,0
3,Even though I have great interest in Biblical ...,0
4,Im a die hard Dads Army fan and nothing will e...,1


In [12]:
df.shape

(50000, 2)

### 2. NLP Preprocessing

In [13]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [14]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)

    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)

In [15]:
df['clean_text'] = df['review'].apply(preprocess)
df.head()

,review,sentiment,clean_text
0,I grew up (b. 1965) watching and loving the Th...,0,grew b watching loving thunderbird mate school...
1,"When I put this movie in my DVD player, and sa...",0,put movie dvd player sat coke chip expectation...
2,Why do people who do not know what a particula...,0,people know particular time past like feel nee...
3,Even though I have great interest in Biblical ...,0,even though great interest biblical movie bore...
4,Im a die hard Dads Army fan and nothing will e...,1,im die hard dad army fan nothing ever change g...


### 3. Feature Engineering

In [16]:
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['clean_text']).toarray()

y = df['sentiment']

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### 4. Model Building


#### 1. Logistic Regression

In [18]:
lr = LogisticRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

#### 2. Naive Bayes

In [19]:
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

#### 3. Decision Tree

In [20]:
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

### 5. Model Evaluation

In [21]:
def evaluate(y_test, y_pred):
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall:", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

In [22]:
print("Logistic Regression")
evaluate(y_test, y_pred_lr)

print("\nNaive Bayes")
evaluate(y_test, y_pred_nb)

print("\nDecision Tree")
evaluate(y_test, y_pred_dt)

Logistic Regression
Accuracy: 0.8863
Precision: 0.8863345003662384
Recall: 0.8863
F1 Score: 0.8862904299047365

Naive Bayes
Accuracy: 0.8564
Precision: 0.8564022021148685
Recall: 0.8564
F1 Score: 0.8564007927390974

Decision Tree
Accuracy: 0.7229
Precision: 0.7230065058768318
Recall: 0.7229
F1 Score: 0.7229034499027622


### 6. Comparison & Insights

In [23]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Naive Bayes', 'Decision Tree'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt)
    ]
})

results

,Model,Accuracy
0,Logistic Regression,0.8863
1,Naive Bayes,0.8564
2,Decision Tree,0.7229


### Final Conclusion:
- NLP preprocessing helped clean noisy text data.
- TF-IDF performed better than Bag of Words.
- Logistic Regression gave the best performance.
- Naive Bayes was faster but slightly less accurate.
- Decision Tree showed overfitting behavior.